|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">A deep understanding of AI language model mechanisms</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Large language models<h1>|
|<h2>Section:</h2>|<h1>Build a GPT<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: Code Attention manually and in Pytorch<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">udemy.com/course/dullms_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import time

# Exercise 1: Simulate data and attention matrices

In [ ]:
# parameters
n_batch = 4
n_embed = 10
context_length = 8
vocab_size = 40

# input data. 
data = torch.randint(vocab_size,(n_batch,context_length)) # [batch,tokens]
    # They are random integers drawn between zero and vocab size which is 40.
    # Size = Batch size x Context length = 4 x 8

In [3]:
# embedding matrix
embeddings = nn.Embedding(vocab_size,n_embed)

# create the q,k,v matrices
key   = nn.Linear(n_embed,n_embed,bias=False)
query = nn.Linear(n_embed,n_embed,bias=False)
value = nn.Linear(n_embed,n_embed,bias=False)

### Process the data

In [4]:
# tokens to embeddings
x = embeddings(data)

# weight the data pre-attention
k = key(x)
q = query(x)
v = value(x)

In [5]:
# print data sizes
print(f'      Data matrix: {data.shape}')
print(f'Embeddings matrix: {embeddings.weight.shape}')
print(f' Token embeddings: {x.shape}')

# sizes of matrices
print('')
print(f'        Size of Q: {key.weight.shape}')
print(f'        Size of K: {key.weight.shape}')
print(f'        Size of V: {key.weight.shape}')

# print attention matrices sizes
print('')
print(f'     Size of Q(x): {k.shape}')
print(f'     Size of K(x): {q.shape}')
print(f'     Size of V(x): {v.shape}')

      Data matrix: torch.Size([4, 8])
Embeddings matrix: torch.Size([40, 10])
 Token embeddings: torch.Size([4, 8, 10])

        Size of Q: torch.Size([10, 10])
        Size of K: torch.Size([10, 10])
        Size of V: torch.Size([10, 10])

     Size of Q(x): torch.Size([4, 8, 10])
     Size of K(x): torch.Size([4, 8, 10])
     Size of V(x): torch.Size([4, 8, 10])


# Exercise 2: Implement self-attention

In [ ]:
### manual implementation

# "cosine similarity" between query and keys (note: would actually be cosine similarity if scaled by |q||k| )
qk = q@k.transpose(-2,-1) # transpose non-batch dimensions

# variance-scale the QK
qk_scaled = qk * n_embed**-.5       # Same as  qk / torch.sqrt(n_embed)
                                    # Dividing qk by square root of the embedding dimension.

# apply mask for future tokens
pastmask = torch.tril(torch.ones(n_batch,context_length,context_length))
qk_scaled[pastmask==0] = -torch.inf # equivalent to adding a matrix of zeros/-infs

# softmaxify
qk_softmax = F.softmax(qk_scaled,dim=-1)

# and final attention mechanism
actsManual = qk_softmax @ v

print(f'Shape of activations (manual): {actsManual.shape}') # [batch, context, n_embed]

Shape of activations (manual): torch.Size([4, 8, 10])


In [7]:
# pytorch implementation
actsTorch = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'Shape of activations (PyTorch): {actsTorch.shape}')

Shape of activations (PyTorch): torch.Size([4, 8, 10])


In [8]:
# compare
print(actsManual[0,:,:])
print('')
print(actsTorch[0,:,:])
print('')
print(actsManual[0,:,:]-actsTorch[0,:,:])

print(f'\n\nAre they _exactly_ equal? {torch.equal(actsTorch,actsManual)}')
print(f'Are they "equal"? {torch.allclose(actsTorch,actsManual)}')

tensor([[ 0.0227,  0.4319, -0.8145,  0.8525, -0.2660,  0.9444,  0.7066, -0.7756,
          0.2582, -0.3426],
        [-0.1086, -0.0049, -0.4373,  0.1239, -0.2343,  0.8730,  0.7733, -0.0286,
          0.1972, -0.0311],
        [ 0.0569,  0.1868, -0.5703,  0.1052, -0.1057,  0.8575,  0.8563, -0.1208,
          0.1703,  0.0986],
        [-0.1403,  0.3109, -0.4073,  0.2457,  0.0086,  0.6717,  0.4983, -0.2112,
          0.1002, -0.0668],
        [-0.3691,  0.1359, -1.1197,  0.0551, -0.4617,  0.8799,  0.7144, -0.1570,
         -0.4026, -0.1427],
        [-0.3515,  0.1444, -1.4917,  0.0661, -0.6390,  1.0084,  0.8830, -0.2493,
         -0.5554, -0.1561],
        [-0.3317,  0.2880, -0.5929,  0.1551, -0.2036,  0.6904,  0.5284, -0.1396,
         -0.0701, -0.0836],
        [-0.1455,  0.3496, -0.9336,  0.1503, -0.2955,  0.6922,  0.7167, -0.2695,
         -0.2257, -0.0358]], grad_fn=<SelectBackward0>)

tensor([[ 0.0227,  0.4319, -0.8145,  0.8525, -0.2660,  0.9444,  0.7066, -0.7756,
          0.2582, 

# Exercise 3: CPU computation time

In [9]:
numReps = 50_000

# the manual version
start_time = time.time()
for _ in range(numReps):
  qk = q@k.transpose(-2,-1) * (n_embed**-.5)
  pastmask = torch.tril(torch.ones(n_batch,context_length,context_length))
  qk[pastmask==0] = -torch.inf
  qk = F.softmax(qk,dim=-1)
  activations = qk @ v
print(f'---    Manual: {time.time()-start_time:.3f} sec')

# the optimized version
start_time = time.time()
for _ in range(numReps):
  activations = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'--- Optimized: {time.time()-start_time:.3f} sec')

---    Manual: 3.166 sec
--- Optimized: 3.233 sec


# Exercise 4: GPU computation time

In [10]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

### Using bigger matrices

In [11]:
# parameters
n_batch = 64
n_embed = 1000
context_length = 2048
vocab_size = 50257

# create matrices
data = torch.randint(vocab_size,(n_batch,context_length),dtype=torch.long,device=device)
embedding = nn.Embedding(vocab_size,n_embed,device=device)
key   = nn.Linear(n_embed,n_embed,bias=False,device=device)
query = nn.Linear(n_embed,n_embed,bias=False,device=device)
value = nn.Linear(n_embed,n_embed,bias=False,device=device)

x = embedding(data)
k = key(x)
q = query(x)
v = value(x)

### Now for the test!

In [12]:
numReps = 200

torch.cuda.synchronize() # synchronize the GPU&CPU. good for time-testing, bad for overall performance
start_time = time.time()
for _ in range(numReps):
  qk = q@k.transpose(-2,-1) * (n_embed**-.5)
  pastmask = torch.tril(torch.ones(n_batch,context_length,context_length,device=device))
  qk[pastmask==0] = -torch.inf
  qk = F.softmax(qk,dim=-1)
  activationsM = qk @ v
print(f'--- Manual:  {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsP = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'--- Pytorch: {time.time()-start_time:.3f} sec')

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# some additional optimizations
import torch._dynamo
SDPA_compiled = torch.compile(F.scaled_dot_product_attention)
torch.set_float32_matmul_precision('high')

In [ ]:
# FYI, FlashAttention: https://github.com/Dao-AILab/flash-attention

In [ ]:
numReps = 200

torch.cuda.synchronize() # synchronize the GPU&CPU. good for time-testing, bad for overall performance
start_time = time.time()
for _ in range(numReps):
  qk = q@k.transpose(-2,-1) * (n_embed**-.5)
  pastmask = torch.tril(torch.ones(n_batch,context_length,context_length,device=device))
  qk[pastmask==0] = -torch.inf
  qk = F.softmax(qk,dim=-1)
  activationsM = qk @ v
print(f'--- Manual:  {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsP = F.scaled_dot_product_attention(q,k,v,is_causal=True)
print(f'--- Pytorch: {time.time()-start_time:.3f} sec')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsO = SDPA_compiled(q,k,v,is_causal=True)
print(f'--- Compiled: {time.time()-start_time:.3f} sec')